In [1]:
import os


#(os.getcwd())
#!ls




'/Users/don/DSC510/Capstone'

In [77]:
# config.py

NUM_DECKS = 2
DEFAULT_WAGER = 25
STARTING_BANKROLL = 10000
SHUFFLE_POINT = 0.25
TOTAL_CARDS = 52 * NUM_DECKS
NUM_SHOES_TO_PLAY = 1000

COUNT_VALUES = {
    '2': +1, '3': +1, '4': +1, '5': +1, '6': +1,
    '7': 0, '8': 0, '9': 0,
    '10': -1, 'J': -1, 'Q': -1, 'K': -1, 'A': -1
}

verbose = False



In [78]:
# environment.py
import random
import config

def create_deck():
    suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
    values = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
    single_deck = [(value, suit) for suit in suits for value in values]
    return single_deck * config.NUM_DECKS

def shuffle_deck(deck):
    random.shuffle(deck)
    return deck

def deal_card(deck, running_count):
    # Check if we are near the cut card point (no direct increment of shoe here)
    if len(deck) <= int(config.TOTAL_CARDS * config.SHUFFLE_POINT):
        if config.verbose:
            print("Cut card reached! Reshuffle after this round.")

    card = deck.pop()
    card_val = config.COUNT_VALUES.get(card[0], 0)
    running_count += card_val
    return card, running_count

def deal_initial_hands(deck, running_count):
    # Deal first card to player
    card, running_count = deal_card(deck, running_count)
    player_hand = [card]
    # Deal second card to player
    card, running_count = deal_card(deck, running_count)
    player_hand.append(card)
    # Deal first card to dealer
    card, running_count = deal_card(deck, running_count)
    dealer_hand = [card]
    # Deal second card to dealer
    card, running_count = deal_card(deck, running_count)
    dealer_hand.append(card)
    return player_hand, dealer_hand, running_count

def reshuffle_if_needed(deck, running_count, shoes_played):
    if len(deck) <= int(config.TOTAL_CARDS * config.SHUFFLE_POINT):
        if config.verbose:
            print("Reshuffling deck now...")
        deck[:] = shuffle_deck(create_deck())
        running_count = 0
        shoes_played += 1
        if config.verbose:
            print(f"*** Completed Shoe {shoes_played} ***")
    return running_count, shoes_played

def calculate_hand_value(hand):
    value = 0
    aces_used_as_11 = 0
    card_values = {
        '2':2,'3':3,'4':4,'5':5,'6':6,'7':7,
        '8':8,'9':9,'10':10,'J':10,'Q':10,'K':10
    }

    for card, suit in hand:
        if card == 'A':
            value += 11
            aces_used_as_11 += 1
        else:
            value += card_values[card]

    while value > 21 and aces_used_as_11 > 0:
        value -= 10
        aces_used_as_11 -= 1

    is_soft = aces_used_as_11 > 0
    return value, is_soft

def is_blackjack(hand):
    value, _ = calculate_hand_value(hand)
    return len(hand) == 2 and value == 21

def can_split(hand):
    if len(hand) == 2:
        card_values = {
            '2':2,'3':3,'4':4,'5':5,'6':6,'7':7,
            '8':8,'9':9,'10':10,'J':10,'Q':10,'K':10,'A':11
        }
        return card_values[hand[0][0]] == card_values[hand[1][0]]
    return False

def dealer_turn(dealer_hand, deck, running_count):
    while True:
        dealer_value, is_soft = calculate_hand_value(dealer_hand)
        if config.verbose:
            print(f"Dealer's Hand: {dealer_hand} | Value: {dealer_value}")

        if dealer_value > 17:
            if config.verbose:
                print("Dealer stands.")
            break
        elif dealer_value == 17 and is_soft:
            if config.verbose:
                print("Dealer hits on soft 17.")
            card, running_count = deal_card(deck, running_count)
            dealer_hand.append(card)
        elif dealer_value == 17:
            if config.verbose:
                print("Dealer stands.")
            break
        else:
            if config.verbose:
                print("Dealer hits.")
            card, running_count = deal_card(deck, running_count)
            dealer_hand.append(card)

    dealer_value, _ = calculate_hand_value(dealer_hand)
    return dealer_hand, dealer_value, running_count

def is_pair(hand):
    return len(hand) == 2 and hand[0][0] == hand[1][0]

def get_card_value(card):
    card_rank = card[0]
    if card_rank in ['J','Q','K','10']:
        return 10
    elif card_rank == 'A':
        return 11
    else:
        return int(card_rank)

def dealer_upcard_value(card):
    if card[0] in ['J','Q','K','10']:
        return 10
    elif card[0] == 'A':
        return 11
    else:
        return int(card[0])

def determine_winner(player_value, dealer_value):
    if config.verbose:
        print("\n--- Final Results ---")
    if player_value > 21:
        if config.verbose:
            print("You busted. Dealer wins.")
    elif dealer_value > 21:
        if config.verbose:
            print("Dealer busted. You win!")
    elif player_value > dealer_value:
        if config.verbose:
            print("You win!")
    elif player_value < dealer_value:
        if config.verbose:
            print("Dealer wins.")
    else:
        if config.verbose:
            print("It's a push!")


In [79]:
# strategy.py
import config
from environment import calculate_hand_value, is_pair, get_card_value, dealer_upcard_value, deal_card

def get_player_decision(hand, dealer_card, bankroll, current_wager, can_double, can_split_hand, splits_done, max_splits=3):
    player_value, is_soft = calculate_hand_value(hand)
    if player_value == 21:
        return 'stand'

    dealer_value = dealer_upcard_value(dealer_card)

    # Pair logic
    if is_pair(hand):
        rank = hand[0][0]
        pair_val = get_card_value(hand[0])
        if rank == 'A':
            if can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                if dealer_value in [5,6] and can_double and bankroll >= current_wager:
                    return 'double'
                return 'hit'

        if pair_val == 10:
            return 'stand'

        if pair_val == 9:
            if dealer_value in [2,3,4,5,6,8,9] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'stand'

        if pair_val == 8:
            if can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                if dealer_value in [2,3,4,5,6]:
                    return 'stand'
                else:
                    return 'hit'

        if pair_val == 7:
            if dealer_value in [2,3,4,5,6,7] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 6:
            if dealer_value in [2,3,4,5,6] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 5:
            if dealer_value in range(2,10) and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'

        if pair_val == 4:
            if dealer_value in [5,6] and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 3:
            if dealer_value in range(2,8) and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

        if pair_val == 2:
            if dealer_value in range(2,8) and can_split_hand and splits_done < max_splits and bankroll >= current_wager:
                return 'split'
            else:
                return 'hit'

    # Soft totals
    if is_soft:
        if player_value == 20:
            return 'stand'
        if player_value == 19:
            if dealer_value == 6 and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'stand'
        if player_value == 18:
            if dealer_value in range(2,7) and can_double and bankroll >= current_wager:
                return 'double'
            elif dealer_value in [9,10,11]:
                return 'hit'
            else:
                return 'stand'
        if player_value == 17:
            if dealer_value in [3,4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 16:
            if dealer_value in [4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 15:
            if dealer_value in [4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 14:
            if dealer_value in [5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 13:
            if dealer_value in [5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'

    # Hard totals
    if not is_soft:
        if player_value >= 17:
            return 'stand'
        if player_value == 16:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 15:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 14:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 13:
            if dealer_value in [2,3,4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 12:
            if dealer_value in [4,5,6]:
                return 'stand'
            else:
                return 'hit'
        if player_value == 11:
            if can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 10:
            if dealer_value in range(2,10) and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value == 9:
            if dealer_value in [3,4,5,6] and can_double and bankroll >= current_wager:
                return 'double'
            else:
                return 'hit'
        if player_value <= 8:
            return 'hit'

    return 'hit'

def player_action(deck, initial_hand, dealer_hand, bankroll, wager, running_count, max_splits=3):
    player_hands = [initial_hand]
    wagers = [wager]
    current_hand_index = 0
    splits_done = 0
    d_up = dealer_hand[0]

    while current_hand_index < len(player_hands):
        hand = player_hands[current_hand_index]

        # If hand has only one card, deal another
        if len(hand) == 1:
            card, running_count = deal_card(deck, running_count)
            hand.append(card)

        while True:
            hand_value, _ = calculate_hand_value(hand)
            if hand_value > 21:
                break

            can_double = (len(hand) == 2)
            can_split_hand = is_pair(hand)

            action = get_player_decision(hand, d_up, bankroll, wagers[current_hand_index], can_double, can_split_hand, splits_done, max_splits)

            if action == 'stand':
                break
            elif action == 'hit':
                card, running_count = deal_card(deck, running_count)
                hand.append(card)
            elif action == 'double':
                if can_double and bankroll >= wagers[current_hand_index]:
                    bankroll -= wagers[current_hand_index]
                    wagers[current_hand_index] *= 2
                    card, running_count = deal_card(deck, running_count)
                    hand.append(card)
                    break
                else:
                    # Couldn't double, just hit
                    card, running_count = deal_card(deck, running_count)
                    hand.append(card)
            elif action == 'split':
                if can_split_hand and splits_done < max_splits and bankroll >= wagers[current_hand_index]:
                    bankroll -= wagers[current_hand_index]
                    c1, c2 = hand[0], hand[1]
                    new_hand_1 = [c1]
                    new_hand_2 = [c2]
                    player_hands[current_hand_index] = new_hand_1
                    player_hands.insert(current_hand_index+1, new_hand_2)
                    wagers.append(wagers[current_hand_index])
                    splits_done += 1

                    # Deal new card to first split hand
                    card, running_count = deal_card(deck, running_count)
                    new_hand_1.append(card)
                    continue
                else:
                    # Couldn't split, just hit
                    card, running_count = deal_card(deck, running_count)
                    hand.append(card)

            hand_value, _ = calculate_hand_value(hand)
            if hand_value > 21:
                break
            if action in ['stand','double']:
                break

        current_hand_index += 1

    return player_hands, wagers, bankroll, running_count

def update_bankroll(bankroll, wager, outcome):
    if outcome == 'win':
        return bankroll + (wager * 2)
    elif outcome == 'push':
        return bankroll + wager
    elif outcome == 'lose':
        return bankroll
    else:
        raise ValueError("Invalid outcome provided.")

def initialize_bankroll(initial_amount=config.STARTING_BANKROLL):
    return initial_amount

def place_wager(bankroll, wager_amount=config.DEFAULT_WAGER):
    if wager_amount > bankroll:
        raise ValueError("Wager exceeds available bankroll.")
    return bankroll - wager_amount, wager_amount



In [80]:
# analytics 
import matplotlib.pyplot as plt
import statistics
import config

class DataLogger:
    def __init__(self):
        # Each record will store details about a single hand
        # For example: {"hand_number": int, "outcome": str, "profit": float, "bankroll": float}
        self.records = []
        self.hand_counter = 0

    def log_hand(self, outcome, profit, bankroll):
        # outcome is 'win', 'lose', or 'push'
        # profit is the net amount won or lost this hand
        # bankroll is the player's bankroll after this hand
        self.records.append({
            "hand_number": self.hand_counter,
            "outcome": outcome,
            "profit": profit,
            "bankroll": bankroll
        })
        self.hand_counter += 1

    def get_data(self):
        # Return a copy of the records as a list of dicts
        return self.records[:]

    def get_outcomes(self):
        # Return a list of outcomes for all hands
        return [r["outcome"] for r in self.records]

    def get_profits(self):
        # Return a list of profits per hand
        return [r["profit"] for r in self.records]

    def get_bankroll_history(self):
        # Return bankroll after each hand
        return [r["bankroll"] for r in self.records]

    def get_counts(self):
        # Count wins, losses, pushes
        wins = sum(1 for r in self.records if r["outcome"] == "win")
        losses = sum(1 for r in self.records if r["outcome"] == "lose")
        pushes = sum(1 for r in self.records if r["outcome"] == "push")
        return wins, losses, pushes


def plot_bankroll_over_time(bankroll_history):
    plt.figure(figsize=(10,5))
    plt.plot(range(len(bankroll_history)), bankroll_history, label='Bankroll')
    plt.title("Bankroll Over Time")
    plt.xlabel("Hand Number")
    plt.ylabel("Bankroll")
    plt.grid(True)
    plt.legend()
    plt.show()

def compute_ev_per_hand(profits):
    # EV = average profit per hand
    if not profits:
        return 0.0
    return sum(profits) / len(profits)

def compute_outcome_rates(wins, losses, pushes, total_hands):
    if total_hands == 0:
        return 0.0, 0.0, 0.0
    win_rate = wins / total_hands
    loss_rate = losses / total_hands
    push_rate = pushes / total_hands
    return win_rate, loss_rate, push_rate

def compute_variance(profits):
    if len(profits) <= 1:
        return 0.0
    return statistics.pvariance(profits)

def print_summary(logger: DataLogger):
    # Print a summary of results using the logger data
    records = logger.get_data()
    total_hands = len(records)
    wins, losses, pushes = logger.get_counts()
    profits = logger.get_profits()
    ev = compute_ev_per_hand(profits)
    var = compute_variance(profits)
    win_rate, loss_rate, push_rate = compute_outcome_rates(wins, losses, pushes, total_hands)
    final_bankroll = logger.get_bankroll_history()[-1] if total_hands > 0 else config.STARTING_BANKROLL
    net_profit = final_bankroll - config.STARTING_BANKROLL
    wager = config.DEFAULT_WAGER
    ev_pct = (ev / wager) if wager != 0 else 0.0

    print("\n--- Simulation Summary ---")
    print(f"Total Hands Played: {total_hands}")
    print(f"Wins: {wins} ({win_rate*100:.2f}%)")
    print(f"Losses: {losses} ({loss_rate*100:.2f}%)")
    print(f"Pushes: {pushes} ({push_rate*100:.2f}%)")
    print(f"Final Bankroll: ${final_bankroll:.2f}")
    print(f"Net Profit/Loss: ${net_profit:.2f}")
    print(f"EV per hand: ${ev:.2f} ({ev_pct:.3f}%)")
    print(f"Variance in profits: {var:.2f}")

In [81]:
# main execution
import config
import environment
import strategy
import analytics

def main():
    logger = analytics.DataLogger()
    bankroll = strategy.initialize_bankroll()
    deck = environment.shuffle_deck(environment.create_deck())

    # Initialize local state variables
    running_count = 0
    shoes_played = 0
    hands_played = 0

    # Run until we have completed the desired number of shoes or the bankroll is depleted
    while shoes_played < config.NUM_SHOES_TO_PLAY and bankroll > 0:
        hand_starting_bankroll = bankroll

        try:
            bankroll, wager = strategy.place_wager(bankroll)
        except ValueError:
            # Can't afford a wager, end simulation
            break

        # Deal initial hands (and update running_count)
        player_hand, dealer_hand, running_count = environment.deal_initial_hands(deck, running_count)
        player_value, _ = environment.calculate_hand_value(player_hand)
        dealer_value, _ = environment.calculate_hand_value(dealer_hand)

        # Check dealer blackjack
        if environment.is_blackjack(dealer_hand):
            outcome = 'push' if environment.is_blackjack(player_hand) else 'lose'
            bankroll = strategy.update_bankroll(bankroll, wager, outcome)
            running_count, shoes_played = environment.reshuffle_if_needed(deck, running_count, shoes_played)
            final_profit = bankroll - hand_starting_bankroll
            logger.log_hand(outcome, final_profit, bankroll)
            hands_played += 1
            continue

        # Check player blackjack
        if environment.is_blackjack(player_hand):
            winnings = (wager * 1.5)
            bankroll += (winnings + wager)
            outcome = 'win'
            running_count, shoes_played = environment.reshuffle_if_needed(deck, running_count, shoes_played)
            final_profit = bankroll - hand_starting_bankroll
            logger.log_hand(outcome, final_profit, bankroll)
            hands_played += 1
            continue

        # Player's turn (now returns running_count)
        player_hands, wagers, bankroll, running_count = strategy.player_action(
            deck, player_hand, dealer_hand, bankroll, wager, running_count, max_splits=3
        )

        # Dealer turn if needed
        if any(environment.calculate_hand_value(h)[0] <= 21 for h in player_hands):
            dealer_hand_copy = dealer_hand.copy()
            dealer_hand_copy, dealer_value, running_count = environment.dealer_turn(dealer_hand_copy, deck, running_count)
        else:
            dealer_value, _ = environment.calculate_hand_value(dealer_hand)

        # Determine outcomes
        subhand_outcomes = []
        for i, hand in enumerate(player_hands, start=1):
            hand_value, _ = environment.calculate_hand_value(hand)
            if hand_value <= 21:
                if dealer_value > 21 or hand_value > dealer_value:
                    subhand_outcomes.append('win')
                    bankroll = strategy.update_bankroll(bankroll, wagers[i-1], 'win')
                elif hand_value < dealer_value:
                    subhand_outcomes.append('lose')
                    bankroll = strategy.update_bankroll(bankroll, wagers[i-1], 'lose')
                else:
                    subhand_outcomes.append('push')
                    bankroll = strategy.update_bankroll(bankroll, wagers[i-1], 'push')
            else:
                subhand_outcomes.append('lose')
                bankroll = strategy.update_bankroll(bankroll, wagers[i-1], 'lose')

        running_count, shoes_played = environment.reshuffle_if_needed(deck, running_count, shoes_played)

        # Aggregate final outcome for the round
        if len(subhand_outcomes) == 1:
            final_outcome = subhand_outcomes[0]
        else:
            unique_outcomes = set(subhand_outcomes)
            final_outcome = subhand_outcomes[0] if len(unique_outcomes) == 1 else 'mixed'

        final_profit = bankroll - hand_starting_bankroll
        logger.log_hand(final_outcome, final_profit, bankroll)

        hands_played += 1

    # Simulation ended
    analytics.print_summary(logger)
    bankroll_history = logger.get_bankroll_history()
    if bankroll_history:
        analytics.plot_bankroll_over_time(bankroll_history)

    # Report final counts
    print(f"Total Hands Played: {hands_played}")
    print(f"Shoes Played: {shoes_played}")

if __name__ == "__main__":
    main()


TypeError: deal_initial_hands() takes 1 positional argument but 2 were given